In [1]:
import torch
import pandas as pd
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification)

#### Model, Tokenizer and Device

In [2]:
model_name = (
    "distilbert/"
    "distilbert-base-uncased-finetuned-sst-2-english"
)
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name
)

model.to(device)
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [4]:
print("Device:", device)
print("Model:", model_name)
print("Labels:", model.config.id2label)

Device: cpu
Model: distilbert/distilbert-base-uncased-finetuned-sst-2-english
Labels: {0: 'NEGATIVE', 1: 'POSITIVE'}


#### Test Tokenizer

In [5]:
sample_text = "I really enjoyed this movie."

encoded_input = tokenizer(
    sample_text,
    return_tensors="pt",
    truncation=True,
    max_length=128
)

tokens = tokenizer.convert_ids_to_tokens(
    encoded_input["input_ids"][0]
)

In [6]:
print("Original text:", sample_text)
print("Tokens:", tokens)

print("Input IDs:", encoded_input["input_ids"])
print("Attention mask:", encoded_input["attention_mask"])

Original text: I really enjoyed this movie.
Tokens: ['[CLS]', 'i', 'really', 'enjoyed', 'this', 'movie', '.', '[SEP]']
Input IDs: tensor([[ 101, 1045, 2428, 5632, 2023, 3185, 1012,  102]])
Attention mask: tensor([[1, 1, 1, 1, 1, 1, 1, 1]])


#### Text Classification Function

In [7]:
def classify_text(texts, max_length=128):
    single_input = isinstance(texts, str)
    if single_input:
        texts = [texts]
    
    encoded_batch = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )
    encoded_batch = {
        key: value.to(device)
        for key, value in encoded_batch.items()
    }
    
    with torch.inference_mode():
        outputs = model(**encoded_batch)
        logits = outputs.logits
        
        probabilities = torch.softmax(logits, dim=-1)
        confidence_socres, predicted_ids = torch.max(
            probabilities, dim=-1
        )
    
    probabilities = probabilities.cpu()
    confidence_scores = confidence_socres.cpu()
    predicted_ids = predicted_ids.cpu()
    
    results = []
    for index, text in enumerate(texts):
        predicted_id = int(predicted_ids[index])
        predicted_label = model.config.id2label[predicted_id]
        
        class_probabilities = {
            model.config.id2label[class_id]:
            float(probabilities[index][class_id])
            for class_id in range(model.config.num_labels)
        }
        
        results.append({
            "text": text,
            "label": predicted_label,
            "confidence": float(
                confidence_scores[index]
            ),
            "probabilities": class_probabilities
        })

    if single_input:
        return results[0]

    return results

#### Single Text Inference

In [8]:
text = "I really enjoyed this movie."
result = classify_text(text)

print("Text:", result["text"])
print("Predicted label:", result["label"])
print(f"Confidence: {result['confidence']:.4f}")
print("Class probabilities:")

for label, probability in result["probabilities"].items():
    print(f"{label}: {probability:.4f}")

Text: I really enjoyed this movie.
Predicted label: POSITIVE
Confidence: 0.9999
Class probabilities:
NEGATIVE: 0.0001
POSITIVE: 0.9999


#### Batch Inference

In [9]:
test_texts = [
    "This product is excellent and works perfectly.",
    "The service was terrible and very slow.",
    "I am extremely disappointed with the result.",
    "The movie was wonderful and inspiring."
]

batch_results = classify_text(test_texts)
result_table = pd.DataFrame([
    {
        "Text": result["text"],
        "Prediction": result["label"],
        "Confidence": round(
            result["confidence"],
            4
        )
    }
    for result in batch_results
])

display(result_table)

,Text,Prediction,Confidence
0,This product is excellent and works perfectly.,POSITIVE,0.9999
1,The service was terrible and very slow.,NEGATIVE,0.9997
2,I am extremely disappointed with the result.,NEGATIVE,0.9998
3,The movie was wonderful and inspiring.,POSITIVE,0.9999
